# Домашнее задание №4: Задача регрессии

### Вопросы и задания:
1. Что представляет из себя разведочный анализ данных?
2. Какая основная библиотека Python используется для анализа данных этим способом?
3. Опишите своими словами процесс анализа данных в соответствии с тем, что рассказывает спикер.
4. Решите задачу, предложенную спикером.
5. Что представляет из себя процесс ETL?

### Процесс выполнения:

Я удалил несколько значения из колонки *возраст*.
Заполняем пропуски данных разными методами, в конце сравним результат.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

In [2]:
data = pd.read_csv('./data/L6SouthGermanCredit.asc', sep=r"\s+")
# Словарь переименования столбцов
column_mapping = {
    "laufkont": "статус_счёта",
    "laufzeit": "срок_кредита",
    "moral": "кредитная_история",
    "verw": "цель_кредита",
    "hoehe": "сумма_кредита",
    "sparkont": "сбережения",
    "beszeit": "стаж_работы",
    "rate": "размер_платежа",
    "famges": "семейное_положение",
    "buerge": "поручители",
    "wohnzeit": "срок_проживания",
    "verm": "имущество",
    "alter": "возраст",
    "weitkred": "другие_кредиты",
    "wohn": "жильё",
    "bishkred": "количество_кредитов",
    "beruf": "работа",
    "pers": "иждивенцы",
    "telef": "телефон",
    "gastarb": "иностранный_работник",
    "kredit": "кредитный_риск"
}

# Переименовываем столбцы
data = data.rename(columns=column_mapping)

data.head()

,статус_счёта,срок_кредита,кредитная_история,цель_кредита,сумма_кредита,сбережения,стаж_работы,размер_платежа,семейное_положение,поручители,срок_проживания,имущество,возраст,другие_кредиты,жильё,количество_кредитов,работа,иждивенцы,телефон,иностранный_работник,кредитный_риск
0,1,18,4,2,1049,1,2,4,2,1,4,2,NaN,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,NaN,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,NaN,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,NaN,3,1,2,2,1,1,1,1
4,3,6,4,0,1299,1,3,1,3,1,1,1,NaN,3,2,3,1,1,1,1,1


In [3]:
print(data["возраст"].head())
real_age = [21,36,23,39,74]

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
Name: возраст, dtype: float64


##### Используем простое усреднение

In [4]:
data["возраст"] = data["возраст"].fillna(round(data["возраст"].mean())).astype(int)
data["возраст"].head()

0    36
1    36
2    36
3    36
4    36
Name: возраст, dtype: int32

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

predicted = data.loc[:4, "возраст"].values  # .values превращает в numpy array
# MAE - средняя абсолютная ошибка (в тех же единицах, что и возраст)
mae = mean_absolute_error(real_age, predicted)
print(f"MAE: {mae:.2f}")
# MSE - средний квадрат ошибки
mse = mean_squared_error(real_age, predicted)
print(f"MSE: {mse:.2f}")

MAE: 13.80
MSE: 369.40


##### KNN - по похожим строкам (sklearn)

In [6]:
from sklearn.impute import KNNImputer

data.loc[:4, "возраст"] = np.nan
cols = data.select_dtypes(include="number").columns
data[cols] = KNNImputer(n_neighbors=5).fit_transform(data[cols])
data["возраст"].head()

0    28.6
1    26.6
2    29.4
3    31.8
4    31.0
Name: возраст, dtype: float64

In [7]:
predicted = data.loc[:4, "возраст"].values
mae = mean_absolute_error(real_age, predicted)
print(f"MAE: {mae:.2f}")
mse = mean_squared_error(real_age, predicted)
print(f"MSE: {mse:.2f}")

MAE: 14.72
MSE: 417.58


##### IterativeImputer - регрессия на остальных столбцах

In [8]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

data.loc[:4, "возраст"] = np.nan
data[cols] = IterativeImputer().fit_transform(data[cols])
data["возраст"].head()

0    30.862506
1    30.294836
2    34.194504
3    31.219109
4    35.349535
Name: возраст, dtype: float64

In [9]:
predicted = data.loc[:4, "возраст"].values
mae = mean_absolute_error(real_age, predicted)
print(f"MAE: {mae:.2f}")
mse = mean_squared_error(real_age, predicted)
print(f"MSE: {mse:.2f}")

MAE: 14.64
MSE: 361.91


##### Вывод
У более сложных алгоритмов нет каких-то явных преимуществ в данном случае, а если судить по MAE, то среднее арифметическое и вовсе лучше себя показало.